In [89]:
# import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import  Path
from sklearn.metrics import mean_absolute_error


In [90]:
PROJECT_ROOT = Path.cwd().parent
output_dir = PROJECT_ROOT / "outputs"

In [91]:
#load the demand and forecast datasets from parquet files
demand_df = pd.read_parquet(output_dir / "gb_demand_2009_2025.parquet")
df_forecast = pd.read_parquet(output_dir / "gb_forecast_2021_2026.parquet")

In [92]:
demand_df.head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,...,shortwave_radiation,year,month,day_of_week,hour,minute,day_of_year,quarter,is_weekend,is_holiday
datetime,,,,,,,,,,,,,,,,,,,,,
2009-01-01 00:00:00+00:00,2009-01-01,1,37910,38704,33939,54,1403,0,0,0,...,0.0,2009,1,3,0,0,1,1,0,1
2009-01-01 00:30:00+00:00,2009-01-01,2,38047,38964,34072,53,1403,0,0,0,...,0.0,2009,1,3,0,30,1,1,0,1
2009-01-01 01:00:00+00:00,2009-01-01,3,37380,38651,33615,53,1403,0,0,0,...,0.0,2009,1,3,1,0,1,1,0,1
2009-01-01 01:30:00+00:00,2009-01-01,4,36426,37775,32526,50,1403,0,0,0,...,0.0,2009,1,3,1,30,1,1,0,1
2009-01-01 02:00:00+00:00,2009-01-01,5,35687,37298,31877,50,1403,0,0,0,...,0.0,2009,1,3,2,0,1,1,0,1


In [93]:
sn_data = demand_df[['SETTLEMENT_DATE', 'SETTLEMENT_PERIOD','ND']].copy()
sn_previous_week = sn_data.copy()
sn_previous_week['SETTLEMENT_DATE'] = sn_previous_week['SETTLEMENT_DATE'] + pd.Timedelta(days=7)
sn_previous_week.rename(columns={'ND': 'seasonal_naive'}, inplace=True)

In [94]:
sn_data = sn_data.merge(
    sn_previous_week[['SETTLEMENT_DATE', 'SETTLEMENT_PERIOD', 'seasonal_naive']],
    on=['SETTLEMENT_DATE', 'SETTLEMENT_PERIOD'],
    how='left'
)

In [95]:
sn_data['seasonal_naive'].isnull().sum()

np.int64(404)

In [96]:
sn_eval = sn_data.dropna(subset=['seasonal_naive'])

In [97]:
mae = mean_absolute_error(sn_eval['ND'], sn_eval['seasonal_naive'])
print(mae)

1799.6262335070408


In [98]:
df_forecast.info()

<class 'pandas.core.frame.DataFrame'>
Index: 96593 entries, 0 to 96592
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   Date               96593 non-null  datetime64[ns, UTC]
 1   Settlement_Period  96593 non-null  int64              
 2   Demand_Forecast    96593 non-null  int64              
 3   Demand_Outturn     96593 non-null  int64              
 4   Absolute_Error     96593 non-null  int64              
 5   Publish_Datetime   96593 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](2), int64(4)
memory usage: 5.2 MB


In [99]:
df_forecast = df_forecast.drop_duplicates(["Date", "Settlement_Period"])

In [100]:
print(df_forecast['Date'].min())
print(df_forecast['Date'].max())

2021-04-01 00:00:00+00:00
2026-10-04 00:00:00+00:00


In [101]:
df_forecast['neso_abs_error'] = np.abs(df_forecast['Demand_Forecast'] - df_forecast['Demand_Outturn'])


In [102]:
#remove timezone information from the 'Date' column in df_forecast
df_forecast['Date'] = df_forecast['Date'].dt.tz_localize(None)
df_forecast['Date']

0       2021-04-01
1       2021-04-01
2       2021-04-01
3       2021-04-01
4       2021-04-01
           ...    
96588   2026-10-04
96589   2026-10-04
96590   2026-10-04
96591   2026-10-04
96592   2026-10-04
Name: Date, Length: 96589, dtype: datetime64[ns]

In [103]:
benchmark = df_forecast.merge(
    sn_data[['SETTLEMENT_DATE', 'SETTLEMENT_PERIOD','ND', 'seasonal_naive']],
    left_on=['Date', 'Settlement_Period'],
    right_on=['SETTLEMENT_DATE', 'SETTLEMENT_PERIOD'],
    how='inner'
)

In [104]:
benchmark.head(10)

,Date,Settlement_Period,Demand_Forecast,Demand_Outturn,Absolute_Error,Publish_Datetime,neso_abs_error,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,seasonal_naive
0,2021-04-01,1,21570,19970,1600,2021-03-31 09:45:00+00:00,1600,2021-04-01,1,19994,24093.0
1,2021-04-01,2,21130,20030,1100,2021-03-31 09:45:00+00:00,1100,2021-04-01,2,19999,24259.0
2,2021-04-01,3,21272,20499,773,2021-03-31 09:45:00+00:00,773,2021-04-01,3,20482,24034.0
3,2021-04-01,4,21600,20940,660,2021-03-31 09:45:00+00:00,660,2021-04-01,4,20926,23654.0
4,2021-04-01,5,21478,20682,796,2021-03-31 09:45:00+00:00,796,2021-04-01,5,20685,23254.0
5,2021-04-01,6,20992,20315,677,2021-03-31 09:45:00+00:00,677,2021-04-01,6,20333,22881.0
6,2021-04-01,7,20695,19998,697,2021-03-31 09:45:00+00:00,697,2021-04-01,7,20004,22767.0
7,2021-04-01,8,20464,19747,717,2021-03-31 09:45:00+00:00,717,2021-04-01,8,19755,22413.0
8,2021-04-01,9,20450,19618,832,2021-03-31 09:45:00+00:00,832,2021-04-01,9,19628,22229.0
9,2021-04-01,10,20806,19527,1279,2021-03-31 09:45:00+00:00,1279,2021-04-01,10,19533,22427.0


In [105]:
benchmark['neso_abs_error_official'] = benchmark['Absolute_Error']
benchmark['neso_abs_error_vs_nd'] = np.abs(benchmark['Demand_Forecast'] - benchmark['ND'])
benchmark['seasonal_naive_abs_error'] = np.abs(benchmark['seasonal_naive'] - benchmark['ND'])

In [106]:
benchmark_eval = benchmark.dropna(subset=['seasonal_naive']).copy()

seasonal_mae = benchmark_eval['seasonal_naive_abs_error'].mean()
neso_vs_nd_mae = benchmark_eval['neso_abs_error_vs_nd'].mean()
neso_official_mae = benchmark_eval['neso_abs_error_official'].mean()

print(f"Seasonal Naive MAE: {seasonal_mae:.2f} MW")
print(f"NESO MAE vs ND:     {neso_vs_nd_mae:.2f} MW")
print(f"NESO Official MAE:  {neso_official_mae:.2f} MW")

Seasonal Naive MAE: 2008.38 MW
NESO MAE vs ND:     622.03 MW
NESO Official MAE:  623.35 MW


In [107]:
benchmark['Publish_Datetime'].dt.tz_convert('Europe/London').dt.hour.value_counts().sort_index()

Publish_Datetime
8     28888
9      3167
10    51268
Name: count, dtype: int64

In [108]:
benchmark_eval.groupby(
    benchmark_eval['Date'].dt.year
)[
    ['seasonal_naive_abs_error',
     'neso_abs_error_vs_nd',
     'neso_abs_error_official']
].mean()

,seasonal_naive_abs_error,neso_abs_error_vs_nd,neso_abs_error_official
Date,,,
2021,1712.509169,580.161034,582.903683
2022,2023.187564,623.401051,625.692075
2023,2043.572220,596.711578,597.605561
2024,2010.971020,620.731439,620.982521
2025,2178.670454,678.842592,679.607993


In [109]:
benchmark_results = benchmark_eval[
    [
        'Date',
        'Settlement_Period',
        'ND',
        'seasonal_naive',
        'Demand_Forecast',
        'Demand_Outturn'
    ]
].copy()